In [ ]:
#install packages, you can also use pip install -e .
%pip install ipykernel
%pip install yfinance
%pip install pandas
%pip install swig
%pip install numpy
%pip install wrds
%pip install pyportfolioopt
%pip install datetime
%pip install requests
%pip install pyarrow
%pip install scipy
%pip install matplotlib
%pip install gymnasium
%pip install stable-baselines3
%pip install stockstats
%pip install tensorboard


#Install finrl library
%pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached swig-4.5.0-py3-none-win_amd64.whl.metadata (3.3 kB)
Using cached swig-4.5.0-py3-none-win_amd64.whl (2.6 MB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to C:\Users\fabia\AppData\Local\Temp\pip-req-build-w2qe3ytz
  Resolved https://github.

  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git 'C:\Users\fabia\AppData\Local\Temp\pip-req-build-w2qe3ytz'
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git 'C:\Users\fabia\AppData\Local\Temp\pip-install-baxw2ahn\elegantrl_5ae36a4cbb2f4c5a9a53fb1ed3c14e0b'


In [1]:
#Import packages

import pandas as pd
import numpy as np 
import datetime
import yfinance as yf 
import requests

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

In [7]:
from io import StringIO

response = requests.get(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
df = tables[0]

tickers = (
    df["Symbol"]
    .astype(str)
    .str.replace(".", "-", regex=False)
    .tolist()
)

In [ ]:
#Generate List of S&P-500 Tickers

# Source - https://stackoverflow.com/a/75845569
# Posted by Asclepius, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-07, License - CC BY-SA 4.0
def list_slickcharts_sp500() -> pd.DataFrame:
    # Ref: https://stackoverflow.com/a/75845569/
    url = 'https://www.slickcharts.com/sp500'
    user_agent = 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/111.0'  # Default user-agent fails.
    response = requests.get(url, headers={'User-Agent': user_agent})
    return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]

df = list_slickcharts_sp500() #Schreibt die erstellte Tabelle in einen Datenframe
df.head() #Zeigt die Struktur der Tabelle
print(df.columns.tolist()) #Gibt die Spaltennamen an -> Debugging
tickers = df.index.str.replace(".", "-", regex=False).tolist() # Schreibt die Indizes in eine Liste, sodass nur noch die Ticker da sind
print(tickers) #Gibt die Liste zur Überprüfung aus -> Debugging

In [8]:
#Debugging
ticker_test = {"AAPL", "MSFT"}

In [9]:
#Setup für den Yahoo Downloader. Zeiträume werden später gesplitet
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-09-01' 
TRADE_START_DATE = '2020-09-01' #Eher Test zur Validierung statt Trading
TRADE_END_DATE = '2026-09-07'

In [10]:
#Daten herunterladen mit dem Yahoodownloader über den gesamten Zeitraum und den 503 Tickersymbolen
df_raw = YahooDownloader(start_date= TRAIN_START_DATE,
                         end_date = TRADE_END_DATE,
                         ticker_list = tickers).fetch_data()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (2081883, 8)


In [12]:
#Daten anssehen
df_raw.head()

Price,date,close,high,low,open,volume,tic,day
0,2009-01-02,10.259192,10.297096,9.798034,9.854888,4236220,A,4
1,2009-01-02,2.714298,2.722973,2.547104,2.568639,746015200,AAPL,4
2,2009-01-02,17.199024,17.292149,16.816896,17.189391,13163193,ABT,4
3,2009-01-02,7.287052,7.395878,7.226829,7.395878,3445200,ACGL,4
4,2009-01-02,24.225479,24.319015,23.412449,23.412449,3787400,ACN,4


In [13]:
#Daten mit technischen Indikatoren erweitern
fe = FeatureEngineer(use_technical_indicator=True, #Berechnung der technischen Indikatoren aktivieren
                     tech_indicator_list = INDICATORS, #Vorgefertigte Liste an Idikatoren übergeben
                     use_vix = True, #Volatilitätsindex aktivieren
                     use_turbulence=True, #Turbulance Index aktivieren
                     user_defined_feature = False) #Usereigene Features aus

semi_processed_data = fe.preprocess_data(df_raw)

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (4446, 8)
Successfully added vix
Successfully added turbulence index


In [14]:
semi_processed_data

,date,close,high,low,open,volume,tic,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,10.259192,10.297096,9.798034,9.854888,4236220,A,4,0.000000,10.924280,9.941550,100.000000,66.666667,100.000000,10.259192,10.259192,39.189999,0.000000
1,2009-01-02,2.714298,2.722973,2.547104,2.568639,746015200,AAPL,4,0.000000,10.924280,9.941550,100.000000,66.666667,100.000000,2.714298,2.714298,39.189999,0.000000
2,2009-01-02,17.199024,17.292149,16.816896,17.189391,13163193,ABT,4,0.000000,10.924280,9.941550,100.000000,66.666667,100.000000,17.199024,17.199024,39.189999,0.000000
3,2009-01-02,7.287052,7.395878,7.226829,7.395878,3445200,ACGL,4,0.000000,10.924280,9.941550,100.000000,66.666667,100.000000,7.287052,7.287052,39.189999,0.000000
4,2009-01-02,24.225479,24.319015,23.412449,23.412449,3787400,ACN,4,0.000000,10.924280,9.941550,100.000000,66.666667,100.000000,24.225479,24.225479,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1840225,2026-09-03,76.339996,76.629997,75.650002,75.940002,3419000,XEL,3,-0.772868,80.151001,74.988001,45.083058,-95.258845,28.006981,77.963000,78.853108,14.320000,517.873625
1840226,2026-09-03,162.210007,165.649994,162.089996,164.429993,13755100,XOM,3,2.523331,168.318670,153.272449,57.069039,80.605606,18.661107,158.511440,150.052798,14.320000,517.873625
1840227,2026-09-03,151.774796,153.157822,149.088343,150.939013,1539100,YUM,3,0.371964,157.515577,142.290898,50.117450,27.203235,0.372816,150.027278,152.472783,14.320000,517.873625
1840228,2026-09-03,98.919998,101.809998,98.150002,100.059998,1276000,ZBH,3,1.398924,102.156020,96.699982,54.999653,48.585317,28.185123,98.063667,93.646884,14.320000,517.873625


In [15]:
#DataFrame nach Tag und Ticker sortieren

list_ticker = semi_processed_data["tic"].unique().tolist()
list_date = list(pd.date_range(semi_processed_data['date'].min(),semi_processed_data['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

fully_processed_data = pd.DataFrame(combination, columns=["date","tic"]).merge(semi_processed_data,on=["date", "tic"],how="left")
fully_processed_data = fully_processed_data[fully_processed_data['date'].isin(semi_processed_data['date'])]
fully_processed_data = fully_processed_data.sort_values(['date','tic'])

fully_processed_data = fully_processed_data.fillna(0)

fully_processed_data.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,A,10.259192,10.297096,9.798034,9.854888,4236220.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,10.259192,10.259192,39.189999,0.0
1,2009-01-02,AAPL,2.714298,2.722973,2.547104,2.568639,746015200.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,2.714298,2.714298,39.189999,0.0
2,2009-01-02,ABT,17.199024,17.292149,16.816896,17.189391,13163193.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,17.199024,17.199024,39.189999,0.0
3,2009-01-02,ACGL,7.287052,7.395878,7.226829,7.395878,3445200.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,7.287052,7.287052,39.189999,0.0
4,2009-01-02,ACN,24.225479,24.319015,23.412449,23.412449,3787400.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,24.225479,24.225479,39.189999,0.0


In [16]:
#Speicherung der Daten

#Aufteilung in 2 seperate Listen
train = data_split(fully_processed_data, TRAIN_START_DATE, TRAIN_END_DATE)
trade = data_split(fully_processed_data, TRADE_START_DATE, TRADE_END_DATE)
print(len(train))
print(len(trade))



1215504
624726


In [21]:
train.to_csv("../data/train.csv", index=False)
trade.to_csv("../data/trade.csv", index=False)